In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
#from google.colab import drive
#drive.mount('/content/drive')
#/content/drive/MyDrive/data1.1.tar.gz

In [3]:
# import gdown

# url = "$SWISSDIAL"

# gdown.download(url, quiet=False)

# !tar -xvzf data1.1.tar.gz -C ./SwissDial

In [4]:
# try:
#     from kaggle_secrets import UserSecretsClient
#     user_secrets = UserSecretsClient()
#     os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
# except:
#     print(":)")

In [5]:
labels = {
    0: "AG",
    1: "BE",
    2: "BS",
    3: "GR",
    4: "LU",
    5: "SG",
    6: "VS",
    7: "ZH",
}

id2label = labels
label2id = {v: k for k, v in labels.items()}
num_labels = len(labels)

In [6]:
#!pip install peft
from transformers import WhisperForConditionalGeneration, WhisperForAudioClassification, WhisperProcessor
#from peft import PeftModel, LoraConfig
import torch

model_id = "Flix-AI/flix-swissgerman-full"

processor = WhisperProcessor.from_pretrained(model_id)
# model = WhisperForConditionalGeneration.from_pretrained(
#     model_id, torch_dtype=torch.bfloat16, device_map="auto"
# )
model = WhisperForAudioClassification.from_pretrained(
    model_id,
    num_labels=8,
    label2id=label2id,
    id2label=id2label,
    torch_dtype=torch.bfloat16,
    #device_map="auto",
)

'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/Flix-AI/flix-swissgerman-full/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].


OSError: Can't load processor for 'Flix-AI/flix-swissgerman-full'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'Flix-AI/flix-swissgerman-full' is the correct path to a directory containing a processor_config.json file

In [ ]:
# Freeze encoder
for param in model.parameters():
    param.requires_grad = False

# Train only the new classification layers
for param in model.projector.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
from datasets import load_dataset, Audio

train_dataset = load_dataset("i4ds/swiss-german-city-sentences_train", split="train")
eval_dataset = load_dataset("i4ds/swiss-german-city-sentences_val", split="train")

train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))
eval_dataset = eval_dataset.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
def preprocess(example):
    raw_label = example["dialect_code"]
    if isinstance(raw_label, str):
        raw_label = raw_label.upper()
    return {
        "labels": label2id.get(raw_label, raw_label)
    }

In [ ]:
train_dataset = train_dataset.map(
    preprocess,
    remove_columns=[c for c in train_dataset.column_names if c != "audio"],
    load_from_cache_file=False
)

eval_dataset = eval_dataset.map(
    preprocess,
    remove_columns=[c for c in eval_dataset.column_names if c != "audio"],
    load_from_cache_file=False
)

In [ ]:
class WhisperAudioCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        audios = [f["audio"]["array"] for f in features]
        sampling_rate = features[0]["audio"]["sampling_rate"]
        
        # Batch feature extraction on CPU/GPU
        batch = self.processor(
            audios,
            sampling_rate=sampling_rate,
            return_tensors="pt",
            padding=True,
        )
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return batch

In [ ]:
!pip install evaluate
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
        
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./dialect-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    # processing_class=processor,
    data_collator=WhisperAudioCollator(processor),
    compute_metrics=compute_metrics,
)

In [ ]:
# Train model
train_result = trainer.train()

# Log and save training state and metrics
# trainer.log_metrics("train", train_result.metrics)
# trainer.save_metrics("train", train_result.metrics)
# trainer.save_state()

In [ ]:
# eval_metrics = trainer.evaluate()
# trainer.log_metrics("eval", eval_metrics)
# trainer.save_metrics("eval", eval_metrics)

In [ ]:
output_dir = "./dialect_classifier_model"

# Save model, config, and label mappings
trainer.save_model(output_dir)

# Save feature extractor and processor settings
# processor.save_pretrained(output_dir)